In [26]:
import uproot
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm


In [27]:
def make_df(file_path):
    df = pd.DataFrame()

    tritrig_file = uproot.open(file_path)

    # Open ROOT tree
    tree = tritrig_file['preselection']

    # Access branches
    B_vertex = tree['vertex.']
    B_ele = tree['ele.']
    B_pos = tree['pos.']


    # Get vertex positions
    vertex_pos = B_vertex['vertex.pos_'].array()
    #df['vertex_pos_x'] = np.array(vertex_pos['fX'])
    #df['vertex_pos_y'] = np.array(vertex_pos['fY'])
    #df['vertex_pos_z'] = np.array(vertex_pos['fZ'])

    # Get arrays of the electron and positron energies as measured with Ecal
    df['ele_E_Ecal'] = np.array(B_ele['ele.energy_'])
    df['pos_E_Ecal'] = np.array(B_pos['pos.energy_'])
    # Get number of hits in Ecal clusters
    #df['ele_Ecal_nhits'] = np.array(tree['ele_clu_nhits'])
    #df['pos_Ecal_nhits'] = np.array(tree['pos_clu_nhits'])
    # Get Ecal cluster positions
    df['ele_Ecal_x']= np.array(B_ele['ele.cluster_.x_'])
    df['ele_Ecal_y']= np.array(B_ele['ele.cluster_.y_'])
    df['ele_Ecal_z']= np.array(B_ele['ele.cluster_.z_'])
    df['pos_Ecal_x']= np.array(B_pos['pos.cluster_.x_'])
    df['pos_Ecal_y']= np.array(B_pos['pos.cluster_.y_'])
    df['pos_Ecal_z']= np.array(B_pos['pos.cluster_.z_'])

    # Get number of hits on track
    #df['ele_trk_nhits'] = np.array(B_ele[ 'ele.track_/ele.track_.n_hits_'])
    #df['pos_trk_nhits'] = np.array(B_pos[ 'pos.track_/pos.track_.n_hits_'])
    # Get Electron and Positron momenta
    arr = B_vertex['vertex.p1_'].array()
    df['ele_Px'] = np.array(arr['fX'])
    df['ele_Py'] = np.array(arr['fY'])
    df['ele_Pz'] = np.array(arr['fZ'])
    arr = B_vertex['vertex.p2_'].array()
    df['pos_Px'] = np.array(arr['fX'])
    df['pos_Py'] = np.array(arr['fY'])
    df['pos_Pz'] = np.array(arr['fZ'])
    # Collect track parameters for electrons and positrons
    #df['ele_phi0'] = -np.array(B_ele['ele.track_/ele.track_.phi0_'])
    #df['ele_dp'] = np.array(B_ele['ele.track_/ele.track_.d0_'])
    #df['ele_kappa_o_alpha'] = -np.array(B_ele['ele.track_/ele.track_.omega_'])
    #df['ele_dz'] = -np.array(B_ele['ele.track_/ele.track_.z0_'])
    #df['ele_tan_lambda'] = -np.array(B_ele['ele.track_/ele.track_.tan_lambda_'])
    #df['pos_phi0'] = -np.array(B_pos['pos.track_/pos.track_.phi0_'])
    #df['pos_dp'] = np.array(B_pos['pos.track_/pos.track_.d0_'])
    #df['pos_kappa_o_alpha'] = -np.array(B_pos['pos.track_/pos.track_.omega_'])
    #df['pos_dz'] = -np.array(B_pos['pos.track_/pos.track_.z0_'])
    #df['pos_tan_lambda'] = -np.array(B_pos['pos.track_/pos.track_.tan_lambda_'])

    return df


In [28]:
# Calculate invariant mass and add to dataframe
m_K = 0.493677

def calculate_invariant_mass_Ptot(df):
    # Calculate total electron momentum
    df['e_Ptot'] = np.sqrt(df['ele_Px']**2 + df['ele_Py']**2 + df['ele_Pz']**2)
    # Calculate total positron momentum
    df['p_Ptot'] = np.sqrt(df['pos_Px']**2 + df['pos_Py']**2 + df['pos_Pz']**2)
    df['e_SVT_E_K'] = m_K / np.sqrt ( 1 - ( df.e_Ptot**2 / (df.e_Ptot**2  + m_K**2) ) )
    df['p_SVT_E_K'] = m_K / np.sqrt ( 1 - ( df.p_Ptot**2 / (df.p_Ptot**2  + m_K**2) ) )
    df['M'] = np.sqrt((df.e_SVT_E_K+df.p_SVT_E_K)**2 - (df.ele_Px+df.pos_Px)**2 - (df.ele_Py+df.pos_Py)**2 - (df.ele_Pz+df.pos_Pz)**2)
    df['Ptot'] = df.e_Ptot + df.p_Ptot

    return df

# Data setup for sideband study a

In [31]:
df_data

,ele_E_Ecal,pos_E_Ecal,ele_Ecal_x,ele_Ecal_y,ele_Ecal_z,pos_Ecal_x,pos_Ecal_y,pos_Ecal_z,ele_Px,ele_Py,ele_Pz,pos_Px,pos_Py,pos_Pz
0,0.000000,1.083690,-9999.000000,-9999.000000,-9999.000000,237.230026,-26.399769,1449.723022,0.028213,0.020584,0.416851,0.049858,-0.020248,1.318486
1,1.103837,0.744567,-175.402161,80.120026,1450.272339,334.084747,-64.636528,1450.799561,0.021702,0.067645,1.214557,0.002822,-0.029053,0.765950
2,0.872991,1.246459,-210.781113,42.332947,1450.265625,258.483337,-31.218971,1449.831665,0.024663,0.027152,1.012383,0.030388,-0.025590,1.142337
3,0.630536,0.914967,-189.955963,28.343084,1450.047485,254.370956,-35.785385,1449.943115,0.036832,0.020526,0.936513,-0.000097,-0.026447,0.975835
4,0.000000,0.818739,-9999.000000,-9999.000000,-9999.000000,255.674377,-80.807861,1450.272339,0.090406,0.044361,2.197935,-0.006584,-0.045707,0.902445
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57816,0.000000,2.819635,-9999.000000,-9999.000000,-9999.000000,159.875565,-26.229998,1449.062744,-0.043139,0.037340,0.696098,0.111247,-0.040254,2.510739
57817,1.239543,2.324663,-179.952606,51.544941,1450.161499,169.012177,-37.392509,1449.285522,0.011717,0.049942,1.330704,0.076432,-0.051683,2.096336
57818,0.000000,0.935489,-9999.000000,-9999.000000,-9999.000000,238.428818,-32.874287,1449.723022,0.069656,0.052150,2.709469,-0.010940,-0.023091,0.962822
57819,0.000000,0.823839,-9999.000000,-9999.000000,-9999.000000,251.632019,79.771584,1450.272339,0.085247,-0.046592,1.018918,-0.024280,0.047206,0.843377


In [ ]:
# Create dataframes 

df_data = make_df('/Users/mghrear/data/HPS_data/2021_v8_pass4_run14727/2021_v8_pass4_run14727.root')

df_data = df_data.sample(frac=1, random_state=42).reset_index(drop=True)
df_data_train_bkg = df_data[:30000].reset_index(drop=True)
df_data_test = df_data[30000:].reset_index(drop=True)

df_data_train_bkg = calculate_invariant_mass_Ptot(df_data_train_bkg)
df_data_train_bkg = df_data_train_bkg.loc[(df_data_train_bkg.M < 1.0084829302730138) | (df_data_train_bkg.M > 1.0303173980209851)  ].reset_index(drop=True)
df_data_train_bkg.drop(columns=['M', 'Ptot', 'e_SVT_E_K', 'p_SVT_E_K','p_Ptot','e_Ptot'], inplace=True)


df_data_train_bkg.to_pickle('/Users/mghrear/data/ML_data/patch/BDT_2021_data_limited_sideband_train_bkg.pk')
df_data_test.to_pickle('/Users/mghrear/data/ML_data/patch/BDT_2021_data_limited_sideband_test.pk')



/var/folders/fx/czrkltw953xcpcjd85tf2tmm0000gn/T/ipykernel_43154/2821622770.py:40: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  df['ele_Px'] = np.array(arr['fX'])
/var/folders/fx/czrkltw953xcpcjd85tf2tmm0000gn/T/ipykernel_43154/2821622770.py:41: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  df['ele_Py'] = np.array(arr['fY'])
/var/folders/fx/czrkltw953xcpcjd85tf2tmm0000gn/T/ipykernel_43154/2821622770.py:42: DeprecationWarning: __array__ implementation doesn't accept a copy 

# Data setup for sideband study b

In [30]:
# Create dataframes 

df_data = make_df('/Users/mghrear/data/HPS_data/2021_v8_pass4_run14727/2021_v8_pass4_run14727.root')


df_data = df_data.sample(frac=1, random_state=42).reset_index(drop=True)
df_data_train_bkg = df_data[:15000].reset_index(drop=True)
df_data_test = df_data[15000:].reset_index(drop=True)

df_data_train_bkg = calculate_invariant_mass_Ptot(df_data_train_bkg)
df_data_train_bkg = df_data_train_bkg.loc[(df_data_train_bkg.M < 1.0084829302730138) | (df_data_train_bkg.M > 1.0303173980209851)  ].reset_index(drop=True)
df_data_train_bkg.drop(columns=['M', 'Ptot', 'e_SVT_E_K', 'p_SVT_E_K','p_Ptot','e_Ptot'], inplace=True)


df_data_train_bkg.to_pickle('/Users/mghrear/data/ML_data/patch/BDT_2021_data_limited_sideband_train_bkg_b.pk')
df_data_test.to_pickle('/Users/mghrear/data/ML_data/patch/BDT_2021_data_limited_sideband_test_b.pk')



/var/folders/fx/czrkltw953xcpcjd85tf2tmm0000gn/T/ipykernel_43154/2821622770.py:40: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  df['ele_Px'] = np.array(arr['fX'])
/var/folders/fx/czrkltw953xcpcjd85tf2tmm0000gn/T/ipykernel_43154/2821622770.py:41: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  df['ele_Py'] = np.array(arr['fY'])
/var/folders/fx/czrkltw953xcpcjd85tf2tmm0000gn/T/ipykernel_43154/2821622770.py:42: DeprecationWarning: __array__ implementation doesn't accept a copy 